In [314]:
"""

LICENSE MIT
2020
Guillaume Rozier
Website : http://www.covidtracker.fr
Mail : guillaume.rozier@telecomnancy.net

README:
This file contains scripts that download data from data.gouv.fr and then process it to build many graphes.

The charts are exported to 'charts/images/france'.
Data is download to/imported from 'data/france'.
Requirements: please see the imports below (use pip3 to install them).

"""

"\n\nLICENSE MIT\n2020\nGuillaume Rozier\nWebsite : http://www.covidtracker.fr\nMail : guillaume.rozier@telecomnancy.net\n\nREADME:\nThis file contains scripts that download data from data.gouv.fr and then process it to build many graphes.\n\nThe charts are exported to 'charts/images/france'.\nData is download to/imported from 'data/france'.\nRequirements: please see the imports below (use pip3 to install them).\n\n"

In [315]:
import pandas as pd
import france_data_management as data
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import plotly
PATH = "../../"
import locale
locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')

'fr_FR.UTF-8'

In [316]:
data.download_data()




  0%|          | 0/8 [00:00<?, ?it/s]


 12%|█▎        | 1/8 [00:01<00:10,  1.57s/it]


 38%|███▊      | 3/8 [00:02<00:06,  1.21s/it]


 75%|███████▌  | 6/8 [00:03<00:01,  1.06it/s]


21it [00:03,  1.51it/s]                      


28it [00:10,  1.29it/s]


36it [00:10,  1.83it/s]

In [317]:
df_hosp = data.import_data_hosp_clage().groupby(["jour", "cl_age90"]).sum().reset_index()
df_hosp_nouveaux = data.import_data_new().groupby("jour").sum().reset_index()

df_hosp = df_hosp[df_hosp.cl_age90 == 0].reset_index()

df_tests_viro = data.import_data_tests_sexe()
df_tests_viro = df_tests_viro[df_tests_viro.cl_age90 == 0].reset_index()

In [318]:
df = df_tests_viro.merge(df_hosp_nouveaux, left_on="jour", right_on="jour")

df = df.reset_index()
df["hosp_cas_ratio"] = df.incid_hosp.rolling(window=7).mean()/df.P.rolling(window=7).mean().shift(7) * 100

In [319]:
#df = df[df.jour >= "2020-11-01"]

In [320]:
y1 = df.P.rolling(window=7).mean()/67000000*100000
y2 = df.incid_dc.rolling(window=7).mean().shift(-14)/67000000

coef_normalisation = y1.max()/y2.max()

fig = go.Figure()
hab_dc = 6000000
fig.add_trace(go.Scatter(
    x=df.jour,
    y=y1,
    name="Cas pour 100 k",
    marker_color='rgb(8, 115, 191)',
    fillcolor="rgba(8, 115, 191, 0.3)",
    fill='tozeroy'))
fig.add_trace(go.Scatter(
    x=df.jour,
    y=-y1,
    name="Miroir des cas",
    marker_color='rgba(8, 115, 191, 0.2)',
    line=dict(
        dash="dot")
))
fig.add_trace(go.Scatter(
    x=df.jour,
    y=-y2*coef_normalisation,
    marker_color='black',
    fillcolor="rgba(0,0,0,0.3)",
    name="Décès hospitaliers<br>avancés de 14 j.<br>pour {} Mio".format(hab_dc/1000000),
    fill='tozeroy'))
fig.update_yaxes()
fig.update_layout(
    title={
                        'text': "Cas vs. Décès hospitaliers",
                        'y':0.97,
                        'x':0.5,
                        'xanchor': 'center',
                        'yanchor': 'top'},
    titlefont = dict(
                    size=30),
    annotations = [
                        dict(
                            x=0.5,
                            y=1.12,
                            xref='paper',
                            yref='paper',
                            font=dict(size=14),
                            text="Cas pour 100 000 habitants et décès hospitaliers avancés de 14 j. pour {} Millions d'habitants<br>{} - @GuillaumeRozier - covidtracker.fr".format(hab_dc/1000000, datetime.strptime(df.jour.max(), '%Y-%m-%d').strftime('%d %B %Y')),#'Date : {}. Source : Santé publique France. Auteur : GRZ - covidtracker.fr.'.format(),                    showarrow = False
                            showarrow=False
                        ),
                        ]
)

#name_fig = "cas_hospitalisations_ratio"
#fig.write_image(PATH + "images/charts/france/{}.jpeg".format(name_fig), scale=2, width=900, height=600)
#plotly.offline.plot(fig, filename = PATH + 'images/html_exports/france/{}.html'.format(name_fig), auto_open=False)
        

In [328]:
y1 = df.P.rolling(window=7).mean()/67000000*100000
y2 = df.incid_hosp.rolling(window=7).mean().shift(-7)/67000000

coef_normalisation = y1.max()/y2.max()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.jour,
    y=y1,
    name="Cas pour 100 k",
    marker_color='rgb(8, 115, 191)',
    fillcolor="rgba(8, 115, 191, 0.3)",
    fill='tozeroy'))
fig.add_trace(go.Scatter(
    x=df.jour,
    y=-y2*coef_normalisation,
    name="Adm. hôpital<br>avancées de 7 j.<br>pour {} Mio".format(round(coef_normalisation/100000)),
    marker_color='rgb(209, 102, 21)',
    fillcolor="rgba(209, 102, 21,0.3)",
    fill='tozeroy'))
fig.add_trace(go.Scatter(
    x=df.jour,
    y=-y1,
    name="Miroir des cas",
    marker_color='rgba(8, 115, 191, 0.2)',
    line=dict(
        dash="dot")
))
fig.update_yaxes()
fig.update_layout(
    title={
                        'text': "Cas vs. Admissions à l'hôpital",
                        'y':0.97,
                        'x':0.5,
                        'xanchor': 'center',
                        'yanchor': 'top'},
    titlefont = dict(
                    size=30),
    annotations = [
                        dict(
                            x=0.5,
                            y=1.12,
                            xref='paper',
                            yref='paper',
                            font=dict(size=14),
                            text="Cas pour 100 000 habitants et admissions à l'hôpital avancées de 7 jours pour 1 Million d'habitants<br>{} - @GuillaumeRozier - covidtracker.fr".format(datetime.strptime(df.jour.max(), '%Y-%m-%d').strftime('%d %B %Y')),#'Date : {}. Source : Santé publique France. Auteur : GRZ - covidtracker.fr.'.format(),                    showarrow = False
                            showarrow=False
                        ),
                        ]
)

#name_fig = "cas_hospitalisations_ratio"
#fig.write_image(PATH + "images/charts/france/{}.jpeg".format(name_fig), scale=2, width=900, height=600)
#plotly.offline.plot(fig, filename = PATH + 'images/html_exports/france/{}.html'.format(name_fig), auto_open=False)
        

In [322]:
y1 = df.P.rolling(window=7).mean()/67000000*100000
y2 = df.incid_rea.rolling(window=7).mean().shift(-7)/67000000*5000000

coef_normalisation = y1.max()/y2.max()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.jour,
    y=y1,
    name="Cas pour 100 k",
    marker_color='rgb(8, 115, 191)',
    fillcolor="rgba(8, 115, 191, 0.3)",
    fill='tozeroy'))
fig.add_trace(go.Scatter(
    x=df.jour,
    y=-y1,
    name="Miroir des cas",
    marker_color='rgba(8, 115, 191, 0.2)',
    line=dict(
        dash="dot")
))
fig.add_trace(go.Scatter(
    x=df.jour,
    y=-y2*coef_normalisation,
    name="Adm. soins critiques<br>avancées de 7 j.<br>pour 5 Mio",
    marker_color='rgb(201, 4, 4)',
    fillcolor="rgba(201, 4, 4,0.3)",
    fill='tozeroy'))
fig.update_yaxes()
fig.update_layout(
    title={
                        'text': "Cas vs. Admissions en soins critiques",
                        'y':0.97,
                        'x':0.5,
                        'xanchor': 'center',
                        'yanchor': 'top'},
    titlefont = dict(
                    size=30),
    annotations = [
                        dict(
                            x=0.5,
                            y=1.12,
                            xref='paper',
                            yref='paper',
                            font=dict(size=14),
                            text="Cas pour 100 000 habitants et admissions en soins critiques (avancées de 7j) pour 5 Millions d'habitants<br>{} - @GuillaumeRozier - covidtracker.fr".format(datetime.strptime(df.jour.max(), '%Y-%m-%d').strftime('%d %B %Y')),#'Date : {}. Source : Santé publique France. Auteur : GRZ - covidtracker.fr.'.format(),                    showarrow = False
                            showarrow=False
                        ),
                        ]
)

#name_fig = "cas_hospitalisations_ratio"
#fig.write_image(PATH + "images/charts/france/{}.jpeg".format(name_fig), scale=2, width=900, height=600)
#plotly.offline.plot(fig, filename = PATH + 'images/html_exports/france/{}.html'.format(name_fig), auto_open=False)
        

In [323]:
df_2021 = df[df["jour"] >= "2021-01-01"]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_2021.jour,
    y=df_2021.hosp_cas_ratio,
    line_color="black",
    showlegend=False))

fig.add_trace(go.Scatter(
    x=[df_2021.jour.values[-1]],
    y=[df_2021.hosp_cas_ratio.values[-1]],
    mode="markers",
    marker_size=7,
    line_color="black",
    showlegend=False))

fig.update_yaxes(ticksuffix=" %")
fig.update_layout(
    title={
                        'text': "Proportion des cas qui sont hospitalisés",
                        'y':0.97,
                        'x':0.5,
                        'xanchor': 'center',
                        'yanchor': 'top'},
    titlefont = dict(
                    size=30),
    annotations = [
                        dict(
                            x=0.5,
                            y=1.12,
                            xref='paper',
                            yref='paper',
                            font=dict(size=14),
                            text="Proportion des admissions à l'hôpital par rapport aux cas positifs 7 jours plus tôt<br>{} - @GuillaumeRozier - covidtracker.fr".format(datetime.strptime(df.jour.max(), '%Y-%m-%d').strftime('%d %B %Y')),#'Date : {}. Source : Santé publique France. Auteur : GRZ - covidtracker.fr.'.format(),                    showarrow = False
                            showarrow=False
                        ),
                        ]
)
y=df_2021.hosp_cas_ratio.values[-1]
fig['layout']['annotations'] += (dict(
                x = df.jour.values[-1], y = y, # annotation point
                xref='x1', 
                yref='y1',
                text="<b>{} %</b> des cas<br>sont hospitalisés".format(round(y, 1)),
                xshift=-2,
                yshift=0,
                xanchor="center",
                align='center',
                font=dict(
                    color="rgb(8, 115, 191)",
                    size=12
                    ),
                opacity=1,
                ax=80,
                ay=0,
                arrowcolor="rgb(8, 115, 191)",
                arrowsize=1.5,
                arrowwidth=0.5,
                arrowhead=0,
                showarrow=True
            ),)

name_fig = "cas_hospitalisations_ratio"
fig.write_image(PATH + "images/charts/france/{}.jpeg".format(name_fig), scale=2, width=950, height=600)
plotly.offline.plot(fig, filename = PATH + 'images/html_exports/france/{}.html'.format(name_fig), auto_open=False)
        

'../../images/html_exports/france/cas_hospitalisations_ratio.html'

In [324]:
"""fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(
    x=df.jour,
    y=df.hosp,
    name="Hosp 70-79",),
    secondary_y=False)
fig.add_trace(go.Scatter(
    x=df.jour,
    y=df.P.rolling(window=7).mean(),
    name="cas positifs 70-79",),
    secondary_y=True)"""

'fig = make_subplots(specs=[[{"secondary_y": True}]])\nfig.add_trace(go.Scatter(\n    x=df.jour,\n    y=df.hosp,\n    name="Hosp 70-79",),\n    secondary_y=False)\nfig.add_trace(go.Scatter(\n    x=df.jour,\n    y=df.P.rolling(window=7).mean(),\n    name="cas positifs 70-79",),\n    secondary_y=True)'




36it [00:29,  1.83it/s]